<a href="https://colab.research.google.com/github/javageek2018/AirlineArrivalDelay/blob/Modeling/target_encoding_pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Target Encoding for Origin and Dest

This notebook recomputes `origin_delay_rate` and `dest_delay_rate` using a leakage-safe approach:

- **Training rows:** expanding window — each row receives the mean delay rate of its airport using only flights that occurred **before** that date. No row sees its own label or any future data.
- **Validation/Test rows:** full training period mean per airport (2018–2022 only). Val/test rows never touch their own `ArrDel15`.
- **Cold start:** airports with no prior history (first appearance, or unseen in training) fall back to the global training mean.


**Input:** `flights_all_features/` (train, val, test parquets — already split temporally)
- Train: 2018–2022
- Val: 2023
- Test: 2024

**Output:** `flights_all_features_encoded/` (same structure, with corrected encoding columns)

In [1]:
# Install PySpark (Colab only)
!pip install pyspark -q

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = SparkSession.builder \
    .appName("TargetEncoding") \
    .config("spark.driver.memory", "16g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.0.2


In [4]:
# Paths
DATA_PATH = "/content/drive/MyDrive/OMDS Capstone/Data/flights_all_features"
OUT_PATH  = "/content/drive/MyDrive/OMDS Capstone/Data/flights_all_features_encoded"

train_df = spark.read.parquet(f"{DATA_PATH}/train.parquet")
val_df   = spark.read.parquet(f"{DATA_PATH}/val.parquet")
test_df  = spark.read.parquet(f"{DATA_PATH}/test.parquet")

print("Train rows:", train_df.count())
print("Val rows:  ", val_df.count())
print("Test rows: ", test_df.count())

Train rows: 31149502
Val rows:   6743403
Test rows:  6965246


In [5]:
# Confirm Origin, Dest, ArrDel15, and FlightDate are present
required = ["Origin", "Dest", "ArrDel15", "FlightDate"]
missing  = [c for c in required if c not in train_df.columns]
print("Missing required columns:", missing if missing else "None — all present")
print("Total columns:", len(train_df.columns))

Missing required columns: None — all present
Total columns: 65


In [6]:
# Drop existing (naively computed) encoding columns so we can recompute them
cols_to_drop = ["origin_delay_rate", "dest_delay_rate"]
existing     = [c for c in cols_to_drop if c in train_df.columns]

train_df = train_df.drop(*existing)
val_df   = val_df.drop(*existing)
test_df  = test_df.drop(*existing)

print(f"Dropped {existing} from all splits")

Dropped ['origin_delay_rate', 'dest_delay_rate'] from all splits


In [7]:
# Global mean delay rate from training data
# Used as fallback for cold-start airports (no prior history)
global_mean = train_df.agg(F.mean("ArrDel15")).collect()[0][0]
print(f"Global training delay rate: {global_mean:.4f} ({global_mean*100:.2f}%)")

Global training delay rate: 0.1785 (17.85%)


In [8]:
# OOF expanding window encoding for TRAINING rows
#
# For each flight at airport X on date D:
#   origin_delay_rate = mean(ArrDel15) for all flights at X with FlightDate < D
#
# rowsBetween(unboundedPreceding, -1) excludes the current row and all rows after it
# in the sort order — so each row only sees past flights at that airport.

window_origin = (
    Window.partitionBy("Origin")
    .orderBy("FlightDate")
    .rowsBetween(Window.unboundedPreceding, -1)
)

window_dest = (
    Window.partitionBy("Dest")
    .orderBy("FlightDate")
    .rowsBetween(Window.unboundedPreceding, -1)
)

train_df = train_df.withColumn(
    "origin_delay_rate",
    F.coalesce(F.mean("ArrDel15").over(window_origin), F.lit(global_mean))
)

train_df = train_df.withColumn(
    "dest_delay_rate",
    F.coalesce(F.mean("ArrDel15").over(window_dest), F.lit(global_mean))
)

print("OOF encoding applied to training data")

OOF encoding applied to training data


In [9]:
# Verify training encoding — spot check a few airports
train_df.select("FlightDate", "Origin", "ArrDel15", "origin_delay_rate") \
    .orderBy("Origin", "FlightDate") \
    .show(10, truncate=False)

+-------------------+------+--------+-------------------+
|FlightDate         |Origin|ArrDel15|origin_delay_rate  |
+-------------------+------+--------+-------------------+
|1514764800000000000|ABE   |1       |0.17850908178243105|
|1514764800000000000|ABE   |0       |1.0                |
|1514764800000000000|ABE   |0       |0.5                |
|1514764800000000000|ABE   |1       |0.3333333333333333 |
|1514764800000000000|ABE   |0       |0.5                |
|1514764800000000000|ABE   |0       |0.4                |
|1514851200000000000|ABE   |0       |0.3333333333333333 |
|1514851200000000000|ABE   |1       |0.2857142857142857 |
|1514851200000000000|ABE   |0       |0.375              |
|1514851200000000000|ABE   |0       |0.3333333333333333 |
+-------------------+------+--------+-------------------+
only showing top 10 rows


In [10]:
# Full training mean per airport — used for val and test lookup
# This is computed AFTER the OOF step so it reflects the final training distribution

origin_encoding = (
    train_df.groupBy("Origin")
    .agg(F.mean("ArrDel15").alias("origin_delay_rate"))
)

dest_encoding = (
    train_df.groupBy("Dest")
    .agg(F.mean("ArrDel15").alias("dest_delay_rate"))
)

print("Training encoding tables computed")
print("Unique origin airports:", origin_encoding.count())
print("Unique dest airports:  ", dest_encoding.count())

Training encoding tables computed
Unique origin airports: 382
Unique dest airports:   382


In [11]:
# Apply training encoding to validation set
# Val rows (2023) receive the full 2018-2022 mean — they never touch their own ArrDel15

val_df = val_df.join(origin_encoding, on="Origin", how="left")
val_df = val_df.join(dest_encoding,   on="Dest",   how="left")

# Cold start: airports in val not seen in training get global mean
val_df = val_df.fillna({"origin_delay_rate": global_mean, "dest_delay_rate": global_mean})

print("Val encoding applied")
val_df.select("FlightDate", "Origin", "origin_delay_rate", "Dest", "dest_delay_rate").show(5)

Val encoding applied
+-------------------+------+-------------------+----+-------------------+
|         FlightDate|Origin|  origin_delay_rate|Dest|    dest_delay_rate|
+-------------------+------+-------------------+----+-------------------+
|1672617600000000000|   BGM|0.13694054776219106| DTW|0.13796009153072786|
|1672704000000000000|   BGM|0.13694054776219106| DTW|0.13796009153072786|
|1672790400000000000|   BGM|0.13694054776219106| DTW|0.13796009153072786|
|1672876800000000000|   BGM|0.13694054776219106| DTW|0.13796009153072786|
|1672963200000000000|   BGM|0.13694054776219106| DTW|0.13796009153072786|
+-------------------+------+-------------------+----+-------------------+
only showing top 5 rows


In [12]:
# Apply training encoding to test set
# Test rows (2024) receive the full 2018-2022 mean — same logic as val

test_df = test_df.join(origin_encoding, on="Origin", how="left")
test_df = test_df.join(dest_encoding,   on="Dest",   how="left")

# Cold start: airports in test not seen in training get global mean
test_df = test_df.fillna({"origin_delay_rate": global_mean, "dest_delay_rate": global_mean})

print("Test encoding applied")
test_df.select("FlightDate", "Origin", "origin_delay_rate", "Dest", "dest_delay_rate").show(5)

Test encoding applied
+-------------------+------+-------------------+----+-------------------+
|         FlightDate|Origin|  origin_delay_rate|Dest|    dest_delay_rate|
+-------------------+------+-------------------+----+-------------------+
|1704067200000000000|   BGM|0.13694054776219106| LGA|0.22399868242527862|
|1704153600000000000|   BGM|0.13694054776219106| LGA|0.22399868242527862|
|1704240000000000000|   BGM|0.13694054776219106| LGA|0.22399868242527862|
|1704326400000000000|   BGM|0.13694054776219106| LGA|0.22399868242527862|
|1704412800000000000|   BGM|0.13694054776219106| LGA|0.22399868242527862|
+-------------------+------+-------------------+----+-------------------+
only showing top 5 rows


In [13]:
# Sanity check — confirm no nulls in encoding columns across all splits
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    null_origin = df.filter(F.col("origin_delay_rate").isNull()).count()
    null_dest   = df.filter(F.col("dest_delay_rate").isNull()).count()
    print(f"{name} — null origin_delay_rate: {null_origin}, null dest_delay_rate: {null_dest}")

train — null origin_delay_rate: 0, null dest_delay_rate: 0
val — null origin_delay_rate: 0, null dest_delay_rate: 0
test — null origin_delay_rate: 0, null dest_delay_rate: 0


In [14]:
# Sanity check — encoding range should be between 0 and 1
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df.agg(
        F.min("origin_delay_rate").alias("origin_min"),
        F.max("origin_delay_rate").alias("origin_max"),
        F.mean("origin_delay_rate").alias("origin_mean"),
        F.min("dest_delay_rate").alias("dest_min"),
        F.max("dest_delay_rate").alias("dest_max"),
        F.mean("dest_delay_rate").alias("dest_mean"),
    ).show()
    print(f"^ {name}")

+----------+----------+-------------------+--------+--------+-------------------+
|origin_min|origin_max|        origin_mean|dest_min|dest_max|          dest_mean|
+----------+----------+-------------------+--------+--------+-------------------+
|       0.0|       1.0|0.18155607588254188|     0.0|     1.0|0.18147939726994045|
+----------+----------+-------------------+--------+--------+-------------------+

^ train
+------------------+-------------------+-------------------+-------------------+------------------+-------------------+
|        origin_min|         origin_max|        origin_mean|           dest_min|          dest_max|          dest_mean|
+------------------+-------------------+-------------------+-------------------+------------------+-------------------+
|0.0680791941646405|0.38524590163934425|0.17952764017714148|0.06953179594689028|0.3304691128657687|0.17980912444597907|
+------------------+-------------------+-------------------+-------------------+------------------+--

In [15]:
# Save to Google Drive
train_df.write.mode("overwrite").parquet(f"{OUT_PATH}/train.parquet")
val_df.write.mode("overwrite").parquet(f"{OUT_PATH}/val.parquet")
test_df.write.mode("overwrite").parquet(f"{OUT_PATH}/test.parquet")

print("Saved to:", OUT_PATH)

Saved to: /content/drive/MyDrive/OMDS Capstone/Data/flights_all_features_encoded
